# Embeddings

Alle bisherigen Methoden hatten einen gemeinsamen blinden Fleck, den wir in Skript 04 benannt haben: In einer Bag-of-Words-Darstellung sind *Kanzler* und *Merz* oder *Benzin* und *Sprit* so verschieden, wie zwei Wörter nur sein können. Ähnlichkeit setzt gemeinsame Zeichenketten voraus.

**Embeddings** ersetzen jedes Wort oder jeden Text durch einen dichten Vektor aus einigen hundert Zahlen. Diese Vektoren werden so gelernt, dass Einheiten, die in ähnlichen Kontexten vorkommen, ähnliche Vektoren erhalten. Das ist die *distributionelle Hypothese*: Ein Wort wird durch die Gesellschaft charakterisiert, in der es auftritt. Embedding-Modelle sind neuronale Netze, aber sie sind **Encoder**, keine Generatoren - sie bilden Text auf Vektoren ab und erzeugen selbst keinen Text. Große Sprachmodelle (LLMs) folgen in Skript 07.

Am Ende dieses Skripts kannst du:

- mit **Wortvektoren** verwandte Wörter finden und erkennen, wo sie danebenliegen;
- **Dokumentvektoren** bilden, zuerst durch Mittelwertbildung, dann mit einem **Sentence Transformer**;
- eine **semantische Suche** durchführen, die Treffer ohne gemeinsame Wörter findet;
- die **Zuordnung von Meldungen** (Story Matching) aus Skript 04 und den **Klassifikator** aus Skript 05 mit Embeddings wiederholen und messen, was sich verändert hat.

## Präambel

Wir laden alle Pakete und Module, die wir im Folgenden benötigen.

In [ ]:
# Datentransformation und Visualisierung
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Paket für linguistische Annotation und Wortvektoren
import spacy

# Pakete für Merkmalsextraktion, Klassifikation und Evaluierung
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.pipeline import FeatureUnion, make_pipeline
from sklearn.svm import LinearSVC

# Pakete für Sentence Transformer und die Projektion in zwei Dimensionen
import torch
import umap
from sentence_transformers import SentenceTransformer

# Hilfsfunktionen des Kurses
import textmining as tm

# Darstellungsoptionen für Tabellen und Grafiken
pd.set_option("display.max_colwidth", 90)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

## Daten

Wir arbeiten wieder mit dem Korpus der Schlagzeilen aus den Nachrichten-Feeds, den wir mit `tm.load_corpus()` laden.

In [ ]:
# Lade das Korpus
corpus = tm.load_corpus()

## Wortvektoren

Die deutschen Modelle von spaCy enthalten statische Wortvektoren: einen 300-dimensionalen Vektor pro Wort, gelernt aus einer großen Sammlung deutscher Texte. "Statisch" bedeutet, dass ein Wort überall, wo es vorkommt, denselben Vektor hat.

Wir beginnen mit einer Falle. Das mittlere Modell aus Skript 02 *hat* Vektoren, aber sieh dir an, wie viele:

In [ ]:
# Lade das mittlere Modell und betrachte seine Vektoren
nlp_md = spacy.load("de_core_news_md")
v = nlp_md.vocab.vectors
print(f"de_core_news_md: {v.n_keys:,} Wörter teilen sich {v.shape[0]:,} verschiedene Vektoren")

# Suche die nächsten Nachbarn von "Kanzler"
keys, _, scores = v.most_similar(np.asarray([nlp_md.vocab["Kanzler"].vector]), n=4)
print("nächste Nachbarn von 'Kanzler':", [(nlp_md.vocab.strings[k], round(float(s), 2)) for k, s in zip(keys[0], scores[0])])

Um klein zu bleiben, bildet das mittlere Modell eine halbe Million Wörter auf zwanzigtausend Vektoren ab. Nicht verwandte Wörter *teilen* sich also einen Vektor und erscheinen als perfekte Nachbarn mit Ähnlichkeit 1.00. Für das, wofür `md` die Vektoren intern verwendet, ist das in Ordnung - für die Untersuchung von Wortbedeutungen ist es nutzlos. Prüfe immer, was deine Vektoren sind, bevor du Schlüsse daraus ziehst.

Das große Modell, `de_core_news_lg`, hat für jedes seiner 500.000 Wörter einen eigenen Vektor. Installiere es einmalig mit `python -m spacy download de_core_news_lg`. Wir benötigen hier nur seine Vektoren, daher lassen wir die übrigen Komponenten der Pipeline weg.

In [ ]:
# Lade das große Modell, nur mit den Komponenten, die wir für die Vektoren brauchen
nlp = spacy.load("de_core_news_lg", exclude=["parser", "ner", "lemmatizer", "attribute_ruler"])
vectors = nlp.vocab.vectors
print(f"de_core_news_lg: {vectors.n_keys:,} Wörter, {vectors.shape[0]:,} verschiedene Vektoren mit {vectors.shape[1]} Dimensionen")


# Finde die n nächsten Nachbarn eines Worts (ohne Varianten in anderer Groß-/Kleinschreibung)
def neighbours(word: str, n: int = 10) -> list[str]:
    keys, _, scores = vectors.most_similar(np.asarray([nlp.vocab[word].vector]), n=n * 3)
    seen, out = {word.casefold()}, []
    for key, score in zip(keys[0], scores[0]):
        w = nlp.vocab.strings[key]
        if w.casefold() not in seen:
            seen.add(w.casefold())
            out.append(f"{w} ({score:.2f})")
    return out[:n]


# Zeige die Nachbarn für einige Beispielwörter
pd.DataFrame({w: neighbours(w) for w in ["Kanzler", "Benzin", "Börse", "Wien", "Angriff"]})

Die Nachbarn sind verwandte Wörter, die ein Bag-of-Words-Modell nie miteinander verbinden könnte. Drei Dinge fallen auf:

- **Die Schreibweise schlägt durch.** Viele Nachbarn sind Flexionsformen und Komposita des Suchworts (*Benzins*, *Superbenzin*, *Gegenangriff*). Diese Vektoren wurden mit Information über Wortteile trainiert. Das hilft bei der deutschen Morphologie, bedeutet aber auch, dass Wörter mit gemeinsamem Stamm einander ähneln, egal was sie bedeuten.
- **Verwandt ist nicht dasselbe wie ähnlich.** Die Nachbarn von *Wien* sind andere österreichische Städte, keine Synonyme für Wien. Wörter, die in denselben *Kontexten* vorkommen, liegen nahe beieinander, und Alternativen teilen sich Kontexte. Für die Suche kann das helfen oder schaden.
- **Die Vektoren sind in der Zeit eingefroren.** Sie wurden aus Texten gelernt, die vor diesem Archiv geschrieben wurden. Sie wissen also nichts darüber, wer zuletzt Kanzler geworden ist.

Und noch einmal das Problem der Komposita:

In [ ]:
# Prüfe, ob es für diese Wörter einen Vektor gibt
for word in ["Sprit", "Spritpreis", "Spritpreise", "Spritpreisbremse", "Sachsen-Anhalt"]:
    print(f"{word:18} hat Vektor: {nlp.vocab[word].has_vector}")

Selbst eine halbe Million Wörter ist ein festes Vokabular. Ein neues Kompositum, das in den Nachrichten dieses Herbsts geprägt wurde, hat überhaupt keinen Vektor und trägt nichts bei - und im Deutschen werden ständig neue Komposita gebildet.

## Dokumentvektoren durch Mittelwertbildung

Der einfachste Dokumentvektor ist der Mittelwert seiner Wortvektoren. spaCy berechnet ihn als `doc.vector`.

In [ ]:
# Drei Umschreibungen derselben Meldung und eine Börsenmeldung
headlines = [
    "Kanzler kündigt Entlastung bei Spritpreisen an",
    "Merz will Steuer auf Benzin senken",
    "Regierungschef plant niedrigere Abgaben für Treibstoff",
    "Dax schließt mit deutlichem Minus",
]
docs = [nlp(h) for h in headlines]

# Vergleiche die Ähnlichkeit zur ersten Schlagzeile: TF-IDF gegen gemittelte Wortvektoren
tfidf_demo = TfidfVectorizer().fit(headlines + corpus["title"].tolist())
pd.DataFrame(
    {
        "TF-IDF": cosine_similarity(tfidf_demo.transform(headlines))[0].round(2),
        "gemittelte Wortvektoren": [docs[0].similarity(d) for d in docs],
    },
    index=headlines,
).round(2)

Die ersten drei Schlagzeilen sagen fast dasselbe mit anderen Worten. TF-IDF sieht überhaupt keine Überschneidung. Gemittelte Wortvektoren erkennen den Zusammenhang - sie bewerten aber auch die Börsenmeldung als ziemlich ähnlich, weil die Mittelwertbildung alles in Richtung einer allgemeinen "Nachrichten"-Richtung verwischt. Außerdem ignoriert die Mittelwertbildung die Wortreihenfolge und kann nicht gewichten, welche Wörter wichtig sind.

## Sentence Transformer

Ein **Sentence Transformer** ist ein neuronaler Encoder, der gezielt darauf trainiert wurde, Texte mit gleicher Bedeutung auf nahe beieinanderliegende Vektoren abzubilden. Er liest den ganzen Satz im Kontext, sodass der Beitrag eines Worts von seinen Nachbarn abhängt, und er kommt mit unbekannten Komposita zurecht, indem er sie in bekannte Wortteile zerlegt.

In diesem Kurs verwenden wir `paraphrase-multilingual-MiniLM-L12-v2`: klein, schnell und auf Paraphrasen in über fünfzig Sprachen trainiert, darunter Deutsch. Beim ersten Ausführen wird das Modell (etwa 0,5 GB) vom Hugging Face Hub heruntergeladen, danach wird es aus dem lokalen Cache geladen. Eine Grafikkarte (GPU) hilft, ist aber für ein Korpus dieser Größe nicht nötig.

In [ ]:
# Verwende die Grafikkarte, falls vorhanden, sonst die CPU
device = "cuda" if torch.cuda.is_available() else "cpu"
encoder = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", device=device)
print("läuft auf", device)

# Kodiere die Beispiel-Schlagzeilen und vergleiche sie mit der ersten
E_demo = encoder.encode(headlines, normalize_embeddings=True)
pd.Series((E_demo @ E_demo[0]).round(2), index=headlines, name="Sentence Transformer").to_frame()

Die drei Paraphrasen liegen jetzt deutlich über der nicht verwandten Börsenmeldung. Jede Schlagzeile ist zu einem 384-dimensionalen Einheitsvektor geworden, und die Ähnlichkeit ist wieder ein Skalarprodukt. Das gesamte Korpus zu kodieren dauert nur Sekunden:

In [ ]:
# Kodiere alle Schlagzeilen des Korpus
E = encoder.encode(corpus["title"].tolist(), batch_size=128, normalize_embeddings=True, show_progress_bar=False)
print(E.shape)

## Semantische Suche

Sind alle Schlagzeilen eingebettet, wird die Suche zu einer Nächste-Nachbarn-Suche: Wir betten die Suchanfrage ein und ordnen die Dokumente nach ihrer Ähnlichkeit. Zum Vergleich führen wir dieselbe Anfrage auch mit TF-IDF durch:

In [ ]:
# Erstelle die TF-IDF-Matrix der Schlagzeilen
tfidf = TfidfVectorizer(min_df=1, token_pattern=r"(?u)\b[^\W\d_][^\W\d_-]*(?:-[^\W\d_]+)*\b")
X = tfidf.fit_transform(corpus["title"])


# Zeige die n besten Treffer der Embedding-Suche und der TF-IDF-Suche nebeneinander
def search(query: str, n: int = 6) -> pd.DataFrame:
    emb = E @ encoder.encode([query], normalize_embeddings=True)[0]
    lex = cosine_similarity(tfidf.transform([query]), X).ravel()
    top_e, top_l = emb.argsort()[::-1][:n], lex.argsort()[::-1][:n]
    return pd.DataFrame({
        "Embedding": [f"{emb[i]:.2f}  {corpus['title'].iloc[i]}" for i in top_e],
        "TF-IDF": [f"{lex[i]:.2f}  {corpus['title'].iloc[i]}" for i in top_l],
    })


search("Autofahrer zahlen immer mehr an der Zapfsäule")

Die Anfrage wurde so formuliert, dass sie nur wenige Wörter mit dem Korpus teilt. TF-IDF findet, was zufällig *Autofahrer* oder *mehr* enthält; die Embedding-Suche findet Schlagzeilen über Spritpreise, also das eigentliche Thema. Das ist der zentrale Vorteil von Embeddings: Sie vergleichen **Bedeutung statt Schreibweise**.

Da das Modell mehrsprachig ist, muss die Anfrage nicht einmal auf Deutsch sein:

In [ ]:
search("Oil prices are rising because of the war in the Middle East")

Die englische Anfrage liefert deutsche Schlagzeilen - etwas, das keine wortbasierte Methode kann. Das bedeutet auch, dass der Feed der EZB, den wir in den Skripten 01-05 weglassen mussten, mit den deutschsprachigen Feeds in einem gemeinsamen Raum verglichen werden könnte.

## Story Matching, noch einmal

In Skript 04 haben wir Schlagzeilen des ORF mit TF-IDF Schlagzeilen der tagesschau zugeordnet und festgestellt, dass die meisten ORF-Meldungen kein nahes Gegenstück hatten. Wir wiederholen genau dasselbe Verfahren mit Embeddings und lesen dieselben Ähnlichkeitsbereiche:

In [ ]:
# Wähle die Meldungen von ORF und tagesschau aus und berechne den zeitlichen Abstand in Stunden
orf = corpus.index[corpus["feed"] == "orf_news"].to_numpy()
tag = corpus.index[corpus["feed"] == "tagesschau"].to_numpy()
pub = corpus["published"].dt.tz_localize(None).to_numpy()
gap_h = (pub[tag][None, :] - pub[orf][:, None]) / np.timedelta64(1, "h")

# Berechne die Ähnlichkeiten, ignoriere Paare mit mehr als 36 Stunden Abstand und wähle den besten Partner
S = E[orf] @ E[tag].T
S[np.abs(gap_h) > 36] = 0
best = S.argmax(axis=1)
pairs = pd.DataFrame({
    "similarity": S[np.arange(len(orf)), best],
    "orf": corpus.loc[orf, "title"].to_numpy(),
    "tagesschau": corpus.loc[tag, "title"].to_numpy()[best],
})

# Zeige je Ähnlichkeitsbereich einige zufällige Paare
for lo, hi in [(0.5, 0.6), (0.6, 0.7), (0.7, 0.8), (0.8, 1.01)]:
    band = pairs[(pairs["similarity"] >= lo) & (pairs["similarity"] < hi)]
    print(f"\n--- Ähnlichkeit {lo:.1f}-{hi:.1f}: {len(band)} Paare")
    for _, r in band.sample(min(4, len(band)), random_state=0).iterrows():
        print(f"  ORF: {r['orf']}\n  TS:  {r['tagesschau']}\n")

Zwei Unterschiede zu Skript 04 fallen auf. Erstens ist die **Skala** eine andere: Die Ähnlichkeiten der Embeddings sind durchwegs höher, weil zwei beliebige Schlagzeilen viel Bedeutung teilen ("jemandem ist irgendwo etwas passiert"). Ein Schwellenwert, der für TF-IDF gewählt wurde, lässt sich daher nicht übertragen. Für jede neue Repräsentation muss der Schwellenwert durch Lesen neu kalibriert werden.

Zweitens sind die **Fehler** andere. Falsche Zuordnungen bei TF-IDF hatten ein Wort gemeinsam; falsche Zuordnungen bei Embeddings haben eine *Art von Meldung* gemeinsam - zwei verschiedene Anschläge, zwei verschiedene Wahlen. Die Embeddings finden Paraphrasen, die TF-IDF übersehen hat, und machen Fehler, die TF-IDF nie machen würde. Keines der beiden Verfahren ist einfach besser; ein robustes Zuordnungsverfahren kombiniert oft beide Signale.

## Eine Landkarte des Korpus

**UMAP** projiziert hochdimensionale Vektoren in zwei Dimensionen und versucht dabei, Nachbarn beisammen zu halten. Es ist ein Werkzeug zur Visualisierung: Abstände zwischen weit entfernten Clustern in der Grafik sind nicht aussagekräftig, nur die lokalen Nachbarschaften sind es.

In [ ]:
# Ein fester random_state macht das Layout reproduzierbar, dafür läuft UMAP nur auf
# einem Kern; n_jobs=1 sagt das ausdrücklich, statt eine Warnung auszulösen.
coords = umap.UMAP(n_neighbors=20, min_dist=0.1, metric="cosine", random_state=0, n_jobs=1).fit_transform(E)

# Zeichne die projizierten Schlagzeilen, eingefärbt nach Feed
fig, ax = plt.subplots(figsize=(7.5, 6))
for feed, g in corpus.groupby("feed"):
    ax.scatter(coords[g.index, 0], coords[g.index, 1], s=4, alpha=0.6, color=tm.FEED_COLOURS[feed],
               label=tm.FEEDS.loc[feed, "outlet"])
ax.set(xticks=[], yticks=[], title="Embeddings der Schlagzeilen, mit UMAP projiziert")
ax.legend(markerscale=4, frameon=False)
plt.show()

Vergleiche das mit der LSA-Grafik in Skript 04. Die Wirtschaftsredaktionen bilden immer noch eigene Regionen, aber die allgemeinen Nachrichten-Feeds durchmischen sich jetzt viel stärker: Schlagzeilen von ORF und tagesschau über dieselbe Art von Ereignis landen in denselben Nachbarschaften, auch wenn sie keine Wörter gemeinsam haben.

## Embeddings als Merkmale für einen Klassifikator

Zum Schluss die Aufgabe aus Skript 05. Statt TF-IDF-Merkmalen geben wir das Embedding jeder Schlagzeile in eine logistische Regression, mit derselben zeitlichen Aufteilung in Trainings- und Testdaten. Die logistische Regression kennen wir bereits aus dem Skript `03-data-mining-classification`.

In [ ]:
# Lade Trainings- und Testdaten und kodiere die Schlagzeilen
train, test = tm.subject_task()
E_train = encoder.encode(train["title"].tolist(), normalize_embeddings=True, show_progress_bar=False)
E_test = encoder.encode(test["title"].tolist(), normalize_embeddings=True, show_progress_bar=False)

# Trainiere die logistische Regression auf den Embeddings
clf = LogisticRegression(max_iter=3000, C=4).fit(E_train, train["label"])
pred_emb = clf.predict(E_test)

# Die beste Pipeline aus Skript 05, hier neu trainiert, damit der Vergleich in sich geschlossen ist.

lesson05 = make_pipeline(
    FeatureUnion([
        ("words", TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True,
                                  token_pattern=r"(?u)\b[^\W\d_][^\W\d_-]*(?:-[^\W\d_]+)*\b")),
        ("chars", TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True)),
    ]),
    LinearSVC(C=1),
).fit(train["title"], train["label"])
pred_tfidf = lesson05.predict(test["title"])

# Vergleiche Accuracy und Makro-F1 der beiden Modelle
pd.DataFrame(
    {
        "Skript 05: TF-IDF + SVM": [accuracy_score(test["label"], pred_tfidf), f1_score(test["label"], pred_tfidf, average="macro")],
        "Embeddings + logistische Regression": [accuracy_score(test["label"], pred_emb), f1_score(test["label"], pred_emb, average="macro")],
    },
    index=["Accuracy", "Makro-F1"],
).T.round(3)

Das Embedding-Modell gewinnt deutlich - mit einer einfachen logistischen Regression darüber und ohne Tuning, gegen eine abgestimmte Pipeline. Zwei Gründe erklären das, und sie gelten allgemeiner:

- **Embeddings generalisieren, dünnbesetzte Merkmale lernen auswendig.** Ein Embedding-Modell bringt Wissen aus seinem Vortraining mit und kommt daher mit Wörtern zurecht, die in den gelabelten Daten fehlen. Die Testdaten stammen aus einem späteren Zeitraum mit neuen Meldungen und neuen Namen - genau dort zählt das. TF-IDF-Merkmale kennen nur die Schlagzeilen aus dem Training: Sie erfassen die Namen und Formate, die *damals* die Labels vorhergesagt haben.
- **Sie scheitern auf unterschiedliche Weise** und ergänzen sich daher. Ein Blick auf die Fälle, in denen die beiden Modelle uneins sind, lohnt sich:

In [ ]:
# Wähle die Schlagzeilen aus, bei denen die beiden Modelle unterschiedlich entscheiden
disagree = test.assign(tfidf=pred_tfidf, embedding=pred_emb)
disagree = disagree[disagree["tfidf"] != disagree["embedding"]]
print(f"die beiden Modelle sind sich bei {len(disagree)} von {len(test)} Schlagzeilen der Testdaten uneins")
disagree[["title", "label", "tfidf", "embedding"]].sample(10, random_state=0)

Viele Unstimmigkeiten gehen auf die Mehrdeutigkeit zwischen Ressort und Ort aus Skript 05 zurück, wobei die beiden Modelle auf verschiedenen Seiten landen. Ein Modell, das nur die Schlagzeile sieht, kann weiterhin nicht wissen, welches Ressort eine Meldung verfasst hat, und dieser kleine Encoder wurde für Paraphrasen trainiert, nicht für die Klassifikation von Nachrichten. Größere Encoder oder das Feintuning eines Encoders auf den gelabelten Schlagzeilen würden weiter helfen - zu echten Kosten an Rechenaufwand und Komplexität.

## Zusammenfassung

| Repräsentation | Erfasst | Übersieht |
|---|---|---|
| TF-IDF (Skripten 03-05) | exakte Wörter, Namen, Formate | Synonyme, Paraphrasen, unbekannte Wörter |
| statische Wortvektoren | verwandte Wörter | Kontext, seltene Komposita; eingefrorenes Vokabular |
| gemittelte Wortvektoren | grobe Bedeutung des Dokuments | Wortreihenfolge, welche Wörter wichtig sind |
| Sentence Transformer | Bedeutung im Kontext, über Sprachen hinweg | exakte Namen; Schwellenwert muss neu kalibriert werden |

## Übungen

1. **Kombiniere die Signale.** Hänge das Embedding an die TF-IDF-Merkmale aus Skript 05 an (`scipy.sparse.hstack`) und trainiere ein gemeinsames Modell. Schlägt es beide?
2. **Sprachübergreifende Zuordnung.** Lade das Korpus mit `languages=("de", "en")` und finde zu jeder Pressemitteilung der EZB die ähnlichste deutschsprachige Schlagzeile. Ergeben die Zuordnungen Sinn?
3. **Kalibrierung des Schwellenwerts.** Bewerte 40 Zuordnungen von Meldungen auf Basis der Embeddings als richtig oder falsch und zeichne die Precision in Abhängigkeit vom Schwellenwert der Ähnlichkeit.
4. **Ein größerer Encoder.** Probiere `intfloat/multilingual-e5-base` aus (stelle jedem Text `"query: "` voran, wie es die Modellbeschreibung verlangt). Ist die Suche besser, und was kostet das an Zeit?

## Quellen

* https://spacy.io/models/de
* https://sbert.net/
* https://umap-learn.readthedocs.io/
* https://scikit-learn.org/stable/modules/feature_extraction.html#text-feature-extraction
* https://scikit-learn.org/stable/modules/svm.html
* https://scikit-learn.org/stable/modules/model_evaluation.html